In [14]:
import re
import json

# --- Chỉ cần chỉnh 2 biến này ---
INPUT_PATH = "vlsp2018_hotel/3-VLSP2018-SA-Hotel-test.txt"      # đường dẫn file txt đầu vào
OUTPUT_PATH = "vlsp2018_hotel/3-VLSP2018-SA-Hotel-test.json"  # đường dẫn file json xuất ra
# ----------------------------------

def prettify_category(code: str) -> str:
    domain, cat = code.split('#', 1)
    pretty_cat = cat.replace('&', ' & ').replace('_', ' ').lower()
    pretty_dom = domain.capitalize()
    return f"{pretty_dom} {pretty_cat}"

def load_category_map_from_codes(text: str) -> dict:
    codes = set(re.findall(r'\{([A-Z0-9_]+#[A-Z0-9&_]+),', text))
    return {code: prettify_category(code) for code in codes}

def clean_line(line: str) -> str:
    """
    Loại bỏ:
      - Dấu thăng + số ở đầu (#1, #2, ...)
      - Các dấu '-' hoặc '_' lặp ở đầu
    """
    # xóa #NUMBER
    line = re.sub(r'^#\d+\s*', '', line)
    # xóa các '-' hoặc '_' ở đầu (1 hoặc nhiều)
    line = re.sub(r'^[-_]+\s*', '', line)
    return line

def parse_blocks(text: str, category_map: dict):
    blocks = re.split(r'\n(?=#\d)', text.strip())
    results = []
    for blk in blocks:
        lines = blk.strip().split('\n')
        # Làm sạch từng dòng
        lines = [clean_line(l) for l in lines]
        # Phần text: tất cả lines không bắt đầu với '{'
        text_part = [l for l in lines if not l.startswith('{')]
        # Phần aspects: dòng đầu tiên bắt đầu với '{'
        aspects_part = next((l for l in lines if l.startswith('{')), "")
        # Gom text, xóa dấu '_' còn sót
        raw = ' '.join(text_part)
        sent = re.sub(r'_+', '', raw).strip()
        # Tách các cặp {CATEGORY, polarity}
        pairs = re.findall(r'\{([^,}]+),\s*([^}]+)\}', aspects_part)
        aspects = []
        for code, pol in pairs:
            code = code.strip()
            pol = pol.strip()
            term = code.split('#')[-1].replace('&', ' & ')
            category = category_map.get(code, code)
            m = re.search(re.escape(term), sent, flags=re.IGNORECASE)
            if m:
                start, end = m.start(), m.end()
            else:
                start, end = 0, len(term)
            aspects.append({
                "term": term,
                "category": category,
                "polarity": pol,
            })
        results.append({
            "text": sent,
            "aspects": aspects
        })
    return {"sentence": results}

if __name__ == "__main__":
    # Đọc input
    with open(INPUT_PATH, encoding="utf-8") as f:
        text = f.read()

    # Sinh CATEGORY_MAP tự động
    category_map = load_category_map_from_codes(text)

    # Parse và ghi output
    result = parse_blocks(text, category_map)
    with open(OUTPUT_PATH, "w", encoding="utf-8") as fo:
        json.dump(result, fo, ensure_ascii=False, indent=4)

    print(f"Đã chuyển và lưu kết quả vào: {OUTPUT_PATH}")

Đã chuyển và lưu kết quả vào: vlsp2018_hotel/3-VLSP2018-SA-Hotel-test.json
